# Phase 1 — Base-Paper Demo: Golf Bag Carrier Robot Computer Vision

**Reference:** Akshay Gupta and Nicholas Gloria, *Golf Bag Carrier Robot Computer Vision*, Stanford CS229 project report (2018): https://cs229.stanford.edu/proj2018/report/12.pdf

This notebook demonstrates the main method families described in the paper on your own video:
1. **MobileNet + SSD-style person/object detection** using TorchVision's pretrained SSD-MobileNetV3-Large detector.
2. **K-Means colour clustering** for rough image-region grouping.
3. **Felzenszwalb superpixels** for region segmentation.
4. A combined visual demo and exported outputs.

**Faithfulness note:** this is a *reproduction-inspired demo*, not an exact reproduction. The original project used its own camera/data, hand-coloured terrain labels, and a trained convolution + softmax pixel classifier. The original ~30,000-image dataset is not provided with the report, so we cannot honestly recreate its trained terrain classifier or claim identical results. We show the segmentation methods and clearly label the missing supervised component.

## 1. Install packages

In [ ]:
%pip -q install torch torchvision opencv-python-headless scikit-image scikit-learn pandas matplotlib pillow

## 2. Upload your video

In [ ]:
from google.colab import files
uploaded = files.upload()
video_files = [n for n in uploaded if n.lower().endswith(('.mp4','.mov','.avi','.mkv','.webm'))]
if not video_files:
    raise ValueError("Upload your walking video (MP4/MOV/AVI/MKV/WEBM).")
VIDEO_PATH = video_files[0]
print("Using:", VIDEO_PATH)

## 3. Read video and sample 50 frames

In [ ]:
import cv2, time, math, os, zipfile, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Could not open video.")
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS = cap.get(cv2.CAP_PROP_FPS) or 0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
indices = np.linspace(0, TOTAL-1, min(50, TOTAL), dtype=int)
frames = []
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, bgr = cap.read()
    if ok: frames.append((int(idx), cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
cap.release()
print(f"{W}x{H}, {TOTAL} frames, {FPS:.2f} FPS; sampled {len(frames)}")
if not frames: raise RuntimeError("No readable frames.")

## 4. Base-paper component A — MobileNet SSD person detection

In [ ]:
import torch
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
detector = ssdlite320_mobilenet_v3_large(weights=weights).to(device).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]

det_rows, det_visuals = [], []
for frame_no, rgb in frames:
    x = preprocess(Image.fromarray(rgb)).to(device)
    start = time.perf_counter()
    with torch.no_grad(): pred = detector([x])[0]
    elapsed = time.perf_counter() - start
    # COCO label 1 = person
    keep = (pred["labels"] == 1) & (pred["scores"] >= 0.35)
    boxes = pred["boxes"][keep].cpu().numpy()
    scores = pred["scores"][keep].cpu().numpy()
    vis = rgb.copy()
    for box, score in zip(boxes, scores):
        x1,y1,x2,y2 = box.astype(int)
        cv2.rectangle(vis,(x1,y1),(x2,y2),(255,80,30),3)
        cv2.putText(vis,f"person {score:.2f}",(x1,max(20,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,80,30),2)
    det_visuals.append(vis)
    det_rows.append({"frame_no":frame_no,"person_detected":int(len(boxes)>0),
                     "person_count":len(boxes),"max_confidence":float(scores.max()) if len(scores) else None,
                     "inference_seconds":elapsed})
det_df = pd.DataFrame(det_rows)
display(det_df.head())
print(f"Person detected in {det_df.person_detected.sum()}/{len(det_df)} sampled frames.")
print("This percentage is a sampled-frame detection rate, not accuracy.")

## 5. Base-paper component B — K-Means colour clustering

In [ ]:
from sklearn.cluster import KMeans

def kmeans_segment(rgb, k=5, sample_limit=12000):
    small = cv2.resize(rgb, (min(320,rgb.shape[1]), int(rgb.shape[0]*min(320,rgb.shape[1])/rgb.shape[1])))
    pixels = small.reshape(-1,3).astype(np.float32)
    if len(pixels) > sample_limit:
        rng = np.random.default_rng(42)
        fit_pixels = pixels[rng.choice(len(pixels), sample_limit, replace=False)]
    else: fit_pixels = pixels
    km = KMeans(n_clusters=k, random_state=42, n_init=5).fit(fit_pixels)
    labels = km.predict(pixels).reshape(small.shape[:2])
    palette = np.clip(km.cluster_centers_,0,255).astype(np.uint8)
    seg = palette[labels]
    return cv2.resize(seg,(rgb.shape[1],rgb.shape[0]),interpolation=cv2.INTER_NEAREST), labels

demo_i = min(25, len(frames)-1)
frame_no, rgb = frames[demo_i]
km_img, km_labels = kmeans_segment(rgb, k=5)
fig, ax = plt.subplots(1,2,figsize=(12,5))
ax[0].imshow(rgb); ax[0].set_title(f"Original — frame {frame_no}")
ax[1].imshow(km_img); ax[1].set_title("K-Means colour clusters")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("Note: colour clusters are not semantic terrain labels.")

## 6. Base-paper component C — Felzenszwalb superpixels

In [ ]:
from skimage.segmentation import felzenszwalb, mark_boundaries

small = cv2.resize(rgb, (min(480,rgb.shape[1]), int(rgb.shape[0]*min(480,rgb.shape[1])/rgb.shape[1])))
segments = felzenszwalb(small, scale=100, sigma=0.8, min_size=80)
boundaries = mark_boundaries(small, segments, color=(1,0,0))
fig, ax = plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(small); ax[0].set_title("Original resized")
ax[1].imshow(segments, cmap="nipy_spectral"); ax[1].set_title(f"Felzenszwalb regions: {len(np.unique(segments))}")
ax[2].imshow(boundaries); ax[2].set_title("Region boundaries")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 7. Combined base-demo preview

In [ ]:
show_n = min(6, len(frames))
fig, axes = plt.subplots(show_n, 3, figsize=(13, 4*show_n))
if show_n == 1: axes = np.array([axes])
for i in range(show_n):
    original = frames[i][1]
    km, _ = kmeans_segment(original, k=5)
    axes[i,0].imshow(original); axes[i,0].set_title(f"Original frame {frames[i][0]}")
    axes[i,1].imshow(det_visuals[i]); axes[i,1].set_title("MobileNet SSD person detection")
    axes[i,2].imshow(km); axes[i,2].set_title("K-Means colour regions")
    for a in axes[i]: a.axis("off")
plt.tight_layout(); plt.show()

## 8. Export baseline outputs

In [ ]:
OUT = "/content/base_paper_demo_outputs"
os.makedirs(OUT, exist_ok=True)
det_df.to_csv(os.path.join(OUT,"mobilenet_ssd_detection.csv"),index=False)
pd.DataFrame([{"frame_no":frames[demo_i][0],"kmeans_clusters":5,
               "felzenszwalb_region_count":int(len(np.unique(segments)))}]).to_csv(
                   os.path.join(OUT,"segmentation_demo_summary.csv"),index=False)
# Save representative images
Image.fromarray(det_visuals[demo_i]).save(os.path.join(OUT,"mobilenet_ssd_detection_demo.png"))
Image.fromarray(km_img).save(os.path.join(OUT,"kmeans_demo.png"))
Image.fromarray((boundaries*255).astype(np.uint8)).save(os.path.join(OUT,"felzenszwalb_boundaries_demo.png"))
summary={"reference_paper":"Golf Bag Carrier Robot Computer Vision",
         "sampled_frames":len(frames),"person_detected_frames":int(det_df.person_detected.sum()),
         "sampled_frame_detection_rate_percent":round(100*float(det_df.person_detected.mean()),2),
         "mean_detector_seconds":round(float(det_df.inference_seconds.mean()),4),
         "not_exact_reproduction":True,
         "supervised_terrain_cnn_recreated":False,
         "reason":"Original hand-labelled training data/checkpoint not supplied with report."}
with open(os.path.join(OUT,"baseline_summary.json"),"w") as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
zip_path="/content/Base_Paper_Demo_Outputs.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for name in os.listdir(OUT): z.write(os.path.join(OUT,name),arcname=name)
files.download(zip_path)

## Baseline interpretation
This demonstrates the paper's core *types* of methods: MobileNet SSD-style detection, K-Means, and Felzenszwalb segmentation. It does not recreate the original camera, dataset, trained terrain CNN/softmax model, or physical robot. Save this output as the baseline and do not compare it with the improved system until both run on the same sampled frames and the same evaluation protocol.